# Case Study 7: Loan Default Prediction — German Credit Dataset
## NBFC Credit Risk Modelling | RBI Compliant | Decision Trees & Ensembles

**Dataset:** UCI Statlog German Credit Data — 1,000 applicants, 20 features, ~30% default rate  
**Source:** https://archive.ics.uci.edu/ml/datasets/statlog+(german+credit+data)  
**Target:** `default` (1 = Bad Credit, 0 = Good Credit)

> **Data Leakage Note:** The German Credit dataset contains only features known at **application time**.  
> No future-leaking columns (unlike LendingClub which has `recovery_amount`, `total_rec_prncp`, `last_pymnt_amnt` etc.).  
> All 20 attributes are safe to use as predictors.

---
### How to run this notebook
1. **Colab / Kaggle:** Upload `german_credit.csv` then change cell 0 to `df = pd.read_csv('german_credit.csv')`  
2. **Any environment:** Run all cells as-is — the dataset is reconstructed from UCI attribute distributions in Cell 0
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
print("Libraries loaded successfully")

---
## Cell 1 — Download Real UCI German Credit Dataset

Downloads the **actual** 1,000-row UCI Statlog German Credit data directly from UCI.
No file upload needed — just run this cell.

In [ ]:
# ── Download REAL UCI German Credit Dataset ──────────────────────────────────
# Source: UCI Machine Learning Repository (official)
# https://archive.ics.uci.edu/ml/datasets/statlog+(german+credit+data)
#
# The raw file uses numeric codes (A11, A12 etc.) — we decode them below.

import urllib.request

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
print("Downloading real UCI German Credit dataset...")
urllib.request.urlretrieve(url, "german.data")
print("Downloaded successfully!")

# Official column names per UCI documentation
col_names = [
    'checking_account', 'duration_months', 'credit_history', 'purpose',
    'credit_amount', 'savings_account', 'employment_length', 'installment_rate',
    'personal_status', 'other_debtors', 'residence_years', 'property', 'age',
    'other_installment_plans', 'housing', 'num_existing_credits', 'job',
    'num_dependents', 'telephone', 'foreign_worker', 'credit_risk'
]

df_raw = pd.read_csv('german.data', sep=' ', header=None, names=col_names)

# Target: 1=Good Credit → 0, 2=Bad Credit → 1
df_raw['default'] = (df_raw['credit_risk'] == 2).astype(int)
df = df_raw.drop(columns=['credit_risk'])

print(f"Shape         : {df.shape}")
print(f"Good Credit   : {(df['default']==0).sum()}")
print(f"Bad Credit    : {(df['default']==1).sum()}")
print(f"Default Rate  : {df['default'].mean():.1%}  (UCI original = 30%)")
df.head()

---
## Cell 2 — Decode UCI Attribute Codes

The raw UCI file uses codes like `A11`, `A12` etc.
This cell maps them to readable labels using the official UCI data dictionary.

In [ ]:
# ── Decode UCI attribute codes → readable labels ─────────────────────────────
# Source: Official UCI German Credit data dictionary

checking_map = {
    'A11':'<0', 'A12':'0_to_200', 'A13':'>200', 'A14':'no_checking'
}
credit_history_map = {
    'A30':'no_credits', 'A31':'all_paid', 'A32':'existing_paid',
    'A33':'delay_in_past', 'A34':'critical_account'
}
purpose_map = {
    'A40':'car_new', 'A41':'car_used', 'A42':'furniture', 'A43':'radio_tv',
    'A44':'appliances', 'A45':'repairs', 'A46':'education', 'A47':'retraining',
    'A48':'business', 'A49':'others', 'A410':'others'
}
savings_map = {
    'A61':'<100', 'A62':'100_to_500', 'A63':'500_to_1000',
    'A64':'>1000', 'A65':'unknown'
}
employment_map = {
    'A71':'unemployed', 'A72':'<1yr', 'A73':'1_to_4yr',
    'A74':'4_to_7yr', 'A75':'>7yr'
}
personal_status_map = {
    'A91':'male_divorced', 'A92':'female_divorced', 'A93':'male_single',
    'A94':'male_married', 'A95':'female_single'
}
other_debtors_map  = {'A101':'none', 'A102':'co-applicant', 'A103':'guarantor'}
property_map       = {'A121':'real_estate', 'A122':'building_society', 'A123':'car', 'A124':'no_property'}
other_plans_map    = {'A141':'bank', 'A142':'stores', 'A143':'none'}
housing_map        = {'A151':'rent', 'A152':'own', 'A153':'free'}
job_map_decode     = {'A171':'unemployed_unskilled', 'A172':'unskilled_resident',
                      'A173':'skilled', 'A174':'highly_qualified'}
telephone_map      = {'A191':0, 'A192':1}
foreign_map        = {'A201':1, 'A202':0}

df['checking_account']        = df['checking_account'].map(checking_map)
df['credit_history']          = df['credit_history'].map(credit_history_map)
df['purpose']                 = df['purpose'].map(purpose_map)
df['savings_account']         = df['savings_account'].map(savings_map)
df['employment_length']       = df['employment_length'].map(employment_map)
df['personal_status']         = df['personal_status'].map(personal_status_map)
df['other_debtors']           = df['other_debtors'].map(other_debtors_map)
df['property']                = df['property'].map(property_map)
df['other_installment_plans'] = df['other_installment_plans'].map(other_plans_map)
df['housing']                 = df['housing'].map(housing_map)
df['job']                     = df['job'].map(job_map_decode)
df['telephone']               = df['telephone'].map(telephone_map)
df['foreign_worker']          = df['foreign_worker'].map(foreign_map)

df.to_csv('german_credit.csv', index=False)
print("Decoded all UCI codes successfully!")
print(f"Checking account values  : {sorted(df['checking_account'].unique())}")
print(f"Credit history values    : {sorted(df['credit_history'].unique())}")
print(f"Employment values        : {sorted(df['employment_length'].unique())}")
df.head()

In [ ]:
# ── UCI Statlog German Credit Dataset ────────────────────────────────────────
# Reconstructed from official UCI attribute frequencies.
# All features are known at loan application time — ZERO DATA LEAKAGE.
# To use the real file instead: df = pd.read_csv('german_credit.csv')

n = 1000
np.random.seed(42)

# Categorical features with documented UCI proportions
checking       = np.random.choice(['no_checking','<0','0_to_200','>200'], n,
                    p=[0.394, 0.274, 0.269, 0.063])
duration       = np.clip(np.random.lognormal(3.0, 0.5, n).astype(int), 4, 72)
credit_history = np.random.choice(
    ['no_credits','all_paid','existing_paid','delay_in_past','critical_account'], n,
    p=[0.040, 0.049, 0.530, 0.088, 0.293])
purpose = np.random.choice(
    ['car_new','car_used','furniture','radio_tv','appliances',
     'repairs','education','retraining','business','others'], n,
    p=[0.234,0.103,0.181,0.280,0.012,0.022,0.050,0.009,0.097,0.012])
credit_amount  = np.clip(np.random.lognormal(7.5, 0.7, n).astype(int), 250, 18424)
savings        = np.random.choice(
    ['unknown','<100','100_to_500','500_to_1000','>1000'], n,
    p=[0.183, 0.603, 0.103, 0.063, 0.048])
employment     = np.random.choice(
    ['unemployed','<1yr','1_to_4yr','4_to_7yr','>7yr'], n,
    p=[0.062, 0.172, 0.339, 0.174, 0.253])
installment_rate = np.random.choice([1,2,3,4], n, p=[0.10,0.231,0.233,0.436])
personal_status  = np.random.choice(
    ['male_divorced','female_divorced','male_single','male_married','female_single'], n,
    p=[0.050, 0.200, 0.548, 0.092, 0.110])
other_debtors    = np.random.choice(['none','co-applicant','guarantor'], n,
                    p=[0.907, 0.041, 0.052])
residence_years  = np.random.choice([1,2,3,4], n, p=[0.130,0.308,0.179,0.383])
property_type    = np.random.choice(
    ['real_estate','building_society','car','no_property'], n,
    p=[0.282, 0.232, 0.332, 0.154])
age              = np.clip(np.random.lognormal(3.6, 0.25, n).astype(int), 19, 75)
other_plans      = np.random.choice(['bank','stores','none'], n, p=[0.139,0.047,0.814])
housing          = np.random.choice(['rent','free','own'], n, p=[0.179,0.108,0.713])
num_credits      = np.random.choice([1,2,3,4], n, p=[0.633,0.272,0.076,0.019])
job              = np.random.choice(
    ['unemployed_unskilled','unskilled_resident','skilled','highly_qualified'], n,
    p=[0.022, 0.200, 0.630, 0.148])
dependents       = np.random.choice([1,2], n, p=[0.845,0.155])
telephone        = np.random.choice([0,1], n, p=[0.404,0.596])
foreign_worker   = np.random.choice([1,0], n, p=[0.963,0.037])

# Calibrated default probability — matches UCI German Credit ~30% rate
logit = np.full(n, -1.2)
ca_map  = {'no_checking':-0.9,'<0':0.9,'0_to_200':0.3,'>200':-0.5}
ch_map  = {'no_credits':0.2,'all_paid':-0.3,'existing_paid':-0.6,
           'delay_in_past':0.5,'critical_account':0.6}
sv_map  = {'unknown':-0.1,'<100':0.6,'100_to_500':0.1,'500_to_1000':-0.2,'>1000':-0.7}
emp_map = {'unemployed':0.5,'<1yr':0.3,'1_to_4yr':0.0,'4_to_7yr':-0.2,'>7yr':-0.4}
pr_map  = {'real_estate':-0.3,'building_society':-0.1,'car':0.1,'no_property':0.4}
job_map = {'unemployed_unskilled':0.4,'unskilled_resident':0.2,
           'skilled':0.0,'highly_qualified':-0.3}
od_map  = {'none':0.0,'co-applicant':-0.2,'guarantor':-0.4}

logit += np.array([ca_map[c]  for c in checking])
logit += np.array([ch_map[c]  for c in credit_history])
logit += np.array([sv_map[s]  for s in savings])
logit += np.array([emp_map[e] for e in employment])
logit += np.array([pr_map[p]  for p in property_type])
logit += np.array([job_map[j] for j in job])
logit += np.array([od_map[o]  for o in other_debtors])
logit += (duration - 20) / 45
logit += (credit_amount - 3000) / 9000
logit += (35 - age) / 70
logit += (installment_rate - 2) * 0.12
logit += np.random.normal(0, 0.25, n)

prob    = 1 / (1 + np.exp(-logit))
default = (np.random.uniform(size=n) < prob).astype(int)

df = pd.DataFrame({
    'checking_account': checking,   'duration_months': duration,
    'credit_history': credit_history,'purpose': purpose,
    'credit_amount': credit_amount,  'savings_account': savings,
    'employment_length': employment, 'installment_rate': installment_rate,
    'personal_status': personal_status,'other_debtors': other_debtors,
    'residence_years': residence_years,'property': property_type,
    'age': age,                      'other_installment_plans': other_plans,
    'housing': housing,              'num_existing_credits': num_credits,
    'job': job,                      'num_dependents': dependents,
    'telephone': telephone,          'foreign_worker': foreign_worker,
    'default': default
})

df.to_csv('german_credit.csv', index=False)

print(f"Shape            : {df.shape}")
print(f"Good Credit (0)  : {(df.default==0).sum()}")
print(f"Bad Credit  (1)  : {(df.default==1).sum()}")
print(f"Default Rate     : {df.default.mean():.1%}  (UCI original ~30%)")
df.head()

---
## Section 1: Exploratory Data Analysis

**Focus areas per case study brief:**
- Default rate by checking account status (strongest predictor in German Credit)
- Default rate by credit history
- Default rate by savings account
- Default rate by employment length
- Age and credit amount distributions: Good vs Bad credit

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('German Credit — EDA: Default Rates by Key Features',
             fontsize=15, fontweight='bold', y=1.01)

# 1. Checking account
ca_order = ['no_checking','>200','0_to_200','<0']
ca_rates = df.groupby('checking_account')['default'].mean().reindex(ca_order) * 100
colors_ca = ['#27AE60','#2ECC71','#F39C12','#E74C3C']
bars = axes[0,0].bar(ca_rates.index, ca_rates.values, color=colors_ca, edgecolor='white')
axes[0,0].axhline(df['default'].mean()*100, color='navy', linestyle='--', alpha=0.7, label='Overall avg')
for bar in bars:
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                   f'{bar.get_height():.1f}%', ha='center', fontsize=9)
axes[0,0].set_title('Default Rate by Checking Account', fontweight='bold')
axes[0,0].set_ylabel('Default Rate (%)'); axes[0,0].legend(fontsize=8)
axes[0,0].tick_params(axis='x', rotation=20)

# 2. Credit history
ch_order = ['critical_account','delay_in_past','no_credits','all_paid','existing_paid']
ch_rates = df.groupby('credit_history')['default'].mean().reindex(ch_order) * 100
colors_ch = ['#E74C3C','#E67E22','#F39C12','#3498DB','#2ECC71']
axes[0,1].barh(ch_order, ch_rates.values, color=colors_ch, edgecolor='white')
axes[0,1].axvline(df['default'].mean()*100, color='navy', linestyle='--', alpha=0.7)
axes[0,1].set_title('Default Rate by Credit History', fontweight='bold')
axes[0,1].set_xlabel('Default Rate (%)')

# 3. Savings account
sv_order = ['<100','100_to_500','500_to_1000','>1000','unknown']
sv_rates = df.groupby('savings_account')['default'].mean().reindex(sv_order) * 100
axes[0,2].bar(sv_order, sv_rates.values,
              color=['#E74C3C','#E67E22','#3498DB','#2ECC71','#95A5A6'], edgecolor='white')
for bar in axes[0,2].patches:
    axes[0,2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                   f'{bar.get_height():.1f}%', ha='center', fontsize=9)
axes[0,2].axhline(df['default'].mean()*100, color='navy', linestyle='--', alpha=0.7)
axes[0,2].set_title('Default Rate by Savings Account', fontweight='bold')
axes[0,2].set_ylabel('Default Rate (%)'); axes[0,2].tick_params(axis='x', rotation=20)

# 4. Employment
emp_order = ['unemployed','<1yr','1_to_4yr','4_to_7yr','>7yr']
emp_rates = df.groupby('employment_length')['default'].mean().reindex(emp_order) * 100
axes[1,0].bar(emp_order, emp_rates.values,
              color=['#E74C3C','#E67E22','#F39C12','#3498DB','#2ECC71'], edgecolor='white')
axes[1,0].axhline(df['default'].mean()*100, color='navy', linestyle='--', alpha=0.7)
axes[1,0].set_title('Default Rate by Employment', fontweight='bold')
axes[1,0].set_ylabel('Default Rate (%)'); axes[1,0].tick_params(axis='x', rotation=15)

# 5. Age distribution
good_age = df.loc[df['default']==0,'age']
bad_age  = df.loc[df['default']==1,'age']
axes[1,1].hist(good_age, bins=25, alpha=0.6, color='#2ECC71', label='Good Credit', density=True)
axes[1,1].hist(bad_age,  bins=25, alpha=0.6, color='#E74C3C', label='Bad Credit',  density=True)
ks_age, _ = stats.ks_2samp(good_age, bad_age)
axes[1,1].set_title('Age: Good vs Bad Credit', fontweight='bold')
axes[1,1].set_xlabel('Age'); axes[1,1].legend(fontsize=9)
axes[1,1].text(0.97, 0.95, f'KS={ks_age:.2f}', transform=axes[1,1].transAxes,
               ha='right', va='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# 6. Credit amount
good_ca = df.loc[df['default']==0,'credit_amount']
bad_ca  = df.loc[df['default']==1,'credit_amount']
axes[1,2].hist(good_ca, bins=30, alpha=0.6, color='#2ECC71', label='Good', density=True)
axes[1,2].hist(bad_ca,  bins=30, alpha=0.6, color='#E74C3C', label='Bad',  density=True)
ks_ca, _ = stats.ks_2samp(good_ca, bad_ca)
axes[1,2].set_title('Credit Amount: Good vs Bad', fontweight='bold')
axes[1,2].set_xlabel('Credit Amount (DM)'); axes[1,2].legend(fontsize=9)
axes[1,2].text(0.97, 0.95, f'KS={ks_ca:.2f}', transform=axes[1,2].transAxes,
               ha='right', va='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('gc_eda.png', bbox_inches='tight', dpi=150)
plt.show()

print("EDA Key Findings:")
print(f"  Checking <0 (overdrawn): {ca_rates.get('<0',0):.1f}% default rate")
print(f"  No checking account    : {ca_rates.get('no_checking',0):.1f}% default rate")
print(f"  Unemployed             : {emp_rates.get('unemployed',0):.1f}% default rate")
print(f"  <100 DM savings        : {sv_rates.get('<100',0):.1f}% default rate")

---
## Section 2: Feature Engineering — No Data Leakage

**Ordinal encoding** preserves natural order for categorical features:

| Feature | Encoding | Rationale |
|---|---|---|
| checking_account | no_checking=0 → >200=3 | Higher balance = lower risk |
| savings_account | <100=1 → >1000=4 | Higher savings = lower risk |
| employment_length | unemployed=0 → >7yr=4 | Longer employment = lower risk |
| credit_history | critical=0 → existing_paid=4 | Better history = lower risk |
| job | unskilled=0 → highly_qualified=3 | Higher skill = lower risk |

**One-hot encoding** for nominal features: purpose, property, housing, other_installment_plans, other_debtors

**Demographics kept separate** for fairness analysis: gender (derived from personal_status), age_bracket

In [ ]:
df_model = df.copy()

# Ordinal encodings — preserve natural risk ordering
checking_ord  = {'no_checking':0,'<0':1,'0_to_200':2,'>200':3}
savings_ord   = {'unknown':0,'<100':1,'100_to_500':2,'500_to_1000':3,'>1000':4}
employment_ord= {'unemployed':0,'<1yr':1,'1_to_4yr':2,'4_to_7yr':3,'>7yr':4}
ch_ord        = {'critical_account':0,'delay_in_past':1,'no_credits':2,
                 'all_paid':3,'existing_paid':4}
job_ord       = {'unemployed_unskilled':0,'unskilled_resident':1,
                 'skilled':2,'highly_qualified':3}

df_model['checking_enc']   = df_model['checking_account'].map(checking_ord)
df_model['savings_enc']    = df_model['savings_account'].map(savings_ord)
df_model['employment_enc'] = df_model['employment_length'].map(employment_ord)
df_model['ch_enc']         = df_model['credit_history'].map(ch_ord)
df_model['job_enc']        = df_model['job'].map(job_ord)

# One-hot encode nominal categoricals
df_model = pd.get_dummies(df_model,
    columns=['purpose','property','housing','other_installment_plans','other_debtors'],
    drop_first=True, dtype=int)

# Demographics (for fairness audit — NOT in primary model)
df_model['is_female']   = df_model['personal_status'].str.startswith('female').astype(int)
df_model['age_bracket'] = pd.cut(df_model['age'], bins=[18,30,45,60,100],
                                  labels=['18-30','31-45','46-60','60+'])

# Final feature set — exclude raw categoricals, demographics, target
exclude = ['checking_account','savings_account','employment_length','credit_history',
           'job','personal_status','age_bracket','default','is_female']
FEATURES = [c for c in df_model.columns if c not in exclude]

X = df_model[FEATURES]
y = df_model['default']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Total features : {len(FEATURES)}")
print(f"Train size     : {X_train.shape}  |  Test size: {X_test.shape}")
print(f"Train default  : {y_train.mean():.3f}  |  Test default: {y_test.mean():.3f}")
print(f"\nFeature list   : {FEATURES}")

---
## Section 3: CART Decision Tree — max_depth Analysis

**CART** (Classification and Regression Trees) uses **Gini Impurity** at each split.

We sweep `max_depth` from 2 to 15 and track train vs test accuracy and ROC-AUC to identify:
- **Underfitting zone**: both train and test accuracy improving
- **Overfitting elbow**: test performance plateaus while train continues rising
- **Overfitting zone**: growing gap between train and test (orange fill)

In [ ]:
depths = list(range(2, 16))
train_acc, test_acc, train_auc, test_auc = [], [], [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42, class_weight='balanced')
    dt.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, dt.predict(X_train)))
    test_acc.append(accuracy_score(y_test,   dt.predict(X_test)))
    train_auc.append(roc_auc_score(y_train, dt.predict_proba(X_train)[:,1]))
    test_auc.append(roc_auc_score(y_test,   dt.predict_proba(X_test)[:,1]))

elbow = depths[np.argmax(test_auc)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('German Credit: Decision Tree Bias-Variance Trade-off', fontsize=14, fontweight='bold')

axes[0].plot(depths, [a*100 for a in train_acc], 'o-', color='#3498DB', label='Train Accuracy', lw=2)
axes[0].plot(depths, [a*100 for a in test_acc],  's-', color='#E74C3C', label='Test Accuracy',  lw=2)
axes[0].axvline(elbow, color='gray', linestyle='--', alpha=0.7, label=f'Elbow ≈ {elbow}')
axes[0].fill_between(depths,
    [a*100 for a in train_acc], [a*100 for a in test_acc],
    alpha=0.12, color='orange', label='Overfit gap')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy vs Depth'); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].plot(depths, train_auc, 'o-', color='#3498DB', label='Train AUC', lw=2)
axes[1].plot(depths, test_auc,  's-', color='#E74C3C', label='Test AUC',  lw=2)
axes[1].axvline(elbow, color='gray', linestyle='--', alpha=0.7, label=f'Elbow ≈ {elbow}')
axes[1].fill_between(depths, train_auc, test_auc, alpha=0.12, color='orange')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('ROC-AUC vs Depth'); axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gc_depth.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Overfitting elbow at max_depth = {elbow}")
print(f"\ndepth | train_acc | test_acc | train_auc | test_auc")
print("-" * 52)
for d,ta,va,tauc,vauc in zip(depths,train_acc,test_acc,train_auc,test_auc):
    flag = " ← ELBOW" if d==elbow else ""
    print(f"  {d:2d}  |  {ta:.4f}   | {va:.4f}   |  {tauc:.4f}   | {vauc:.4f}{flag}")

---
## Section 4: Cost-Complexity Pruning (CCP)

Instead of fixing `max_depth` by eye, **CCP** penalises each extra leaf node by a parameter **α**.  
We find the optimal α using **5-fold stratified cross-validation** on the training set.

- Higher α → simpler tree (fewer leaves)
- Optimal α → maximum CV ROC-AUC

In [ ]:
# Cost-complexity pruning path
dt_full = DecisionTreeClassifier(random_state=42, class_weight='balanced')
path    = dt_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[path.ccp_alphas <= 0.04]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for alpha in ccp_alphas:
    dt_cv  = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42, class_weight='balanced')
    scores = cross_val_score(dt_cv, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_scores.append((alpha, scores.mean(), scores.std()))

cv_df = pd.DataFrame(cv_scores, columns=['alpha','mean_auc','std_auc'])
optimal_alpha = cv_df.loc[cv_df['mean_auc'].idxmax(), 'alpha']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cv_df['alpha'], cv_df['mean_auc'], 'o-', color='#3498DB', lw=2, label='CV Mean AUC')
ax.fill_between(cv_df['alpha'],
    cv_df['mean_auc'] - cv_df['std_auc'],
    cv_df['mean_auc'] + cv_df['std_auc'],
    alpha=0.2, color='#3498DB', label='±1 std')
ax.axvline(optimal_alpha, color='#E74C3C', linestyle='--', lw=2,
           label=f'Optimal α = {optimal_alpha:.5f}')
ax.set_xlabel('Cost-Complexity Parameter (alpha)', fontsize=11)
ax.set_ylabel('Cross-Validated ROC-AUC', fontsize=11)
ax.set_title('German Credit: CCP Pruning — CV AUC vs Alpha', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gc_ccp.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Optimal alpha : {optimal_alpha:.6f}")
print(f"Best CV AUC   : {cv_df['mean_auc'].max():.4f}")

In [ ]:
# Train pruned tree
dt_pruned = DecisionTreeClassifier(
    ccp_alpha=optimal_alpha, random_state=42, class_weight='balanced')
dt_pruned.fit(X_train, y_train)

print(f"Pruned tree depth  : {dt_pruned.get_depth()}")
print(f"Number of leaves   : {dt_pruned.get_n_leaves()}")
print(f"Test Accuracy      : {accuracy_score(y_test, dt_pruned.predict(X_test)):.4f}")
print(f"Test ROC-AUC       : {roc_auc_score(y_test, dt_pruned.predict_proba(X_test)[:,1]):.4f}")
print()
print("TREE RULES (text):")
print(export_text(dt_pruned, feature_names=FEATURES))

# Annotated pruned tree visualisation
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(dt_pruned, feature_names=FEATURES,
          class_names=['Good Credit','Bad Credit'],
          filled=True, rounded=True, ax=ax,
          fontsize=8, impurity=True, proportion=False, precision=3)
ax.set_title(
    f'German Credit — Pruned Decision Tree '
    f'(α={optimal_alpha:.5f}, Depth={dt_pruned.get_depth()}, Leaves={dt_pruned.get_n_leaves()})',
    fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('gc_pruned_tree.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Section 5: Model Comparison — Decision Tree vs Logistic Regression

**KS Statistic** = max(TPR − FPR) — measures separation between good and bad credit score distributions.

| KS Range | Quality |
|---|---|
| < 0.20 | Poor |
| 0.20–0.40 | Acceptable |
| 0.40–0.60 | Good |
| > 0.60 | Very Good |

In [ ]:
# Logistic Regression baseline
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, C=0.5))
])
lr_pipe.fit(X_train, y_train)
lr_proba = lr_pipe.predict_proba(X_test)[:,1]
lr_pred  = lr_pipe.predict(X_test)
dt_proba = dt_pruned.predict_proba(X_test)[:,1]
dt_pred  = dt_pruned.predict(X_test)

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_proba)
auc_lr = roc_auc_score(y_test, lr_proba)
auc_dt = roc_auc_score(y_test, dt_proba)

def ks_statistic(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return np.max(tpr - fpr)

ks_lr = ks_statistic(y_test, lr_proba)
ks_dt = ks_statistic(y_test, dt_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('German Credit: Decision Tree vs Logistic Regression', fontsize=14, fontweight='bold')

axes[0].plot(fpr_lr, tpr_lr, lw=2.5, color='#3498DB',
             label=f'Logistic Regression (AUC={auc_lr:.3f}, KS={ks_lr:.3f})')
axes[0].plot(fpr_dt, tpr_dt, lw=2.5, color='#E67E22',
             label=f'Pruned Decision Tree (AUC={auc_dt:.3f}, KS={ks_dt:.3f})')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison'); axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

metrics   = ['ROC-AUC','KS Statistic','F1-Score','Accuracy']
lr_vals   = [auc_lr, ks_lr, f1_score(y_test,lr_pred), accuracy_score(y_test,lr_pred)]
dt_vals   = [auc_dt, ks_dt, f1_score(y_test,dt_pred), accuracy_score(y_test,dt_pred)]
x = np.arange(len(metrics)); w = 0.35
axes[1].bar(x-w/2, lr_vals, w, color='#3498DB', label='Logistic Regression', edgecolor='white')
axes[1].bar(x+w/2, dt_vals, w, color='#E67E22', label='Pruned Decision Tree', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(metrics, fontsize=10)
axes[1].set_title('Performance Metrics'); axes[1].legend(); axes[1].set_ylim(0,1.05)
for i,(lv,dv) in enumerate(zip(lr_vals,dt_vals)):
    axes[1].text(i-w/2, lv+0.01, f'{lv:.3f}', ha='center', fontsize=8)
    axes[1].text(i+w/2, dv+0.01, f'{dv:.3f}', ha='center', fontsize=8)
axes[1].grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('gc_model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"{'Metric':<20} {'Logistic Reg':>14} {'Decision Tree':>15}")
print("-"*52)
for m,lv,dv in zip(metrics,lr_vals,dt_vals):
    print(f"{m:<20} {lv:>14.4f} {dv:>15.4f}")

---
## Section 6: CHAID vs CART — Conceptual Discussion

| Dimension | CART | CHAID |
|---|---|---|
| Split criterion | Gini Impurity / MSE | Chi-square / F-test |
| Branching | Binary only | Multi-way (≥2 children) |
| Data affinity | Numerical & ordinal | Categorical / nominal |
| Pruning | CCP (post-prune) | Bonferroni correction |
| Statistical test | None (greedy) | Yes — significance based |

**When to choose CHAID for German Credit:**  
The dataset is **heavily categorical** (checking account, credit history, purpose, savings, employment — all categorical).  
CHAID would split `checking_account` into all 4 categories at once instead of forcing binary splits.  
In a regulatory setting, CHAID's chi-square p-values provide statistical justification for each split.

**When to stick with CART:**  
When using sklearn (no CHAID library), for ensemble methods (Random Forest, XGBoost), or when continuous features dominate.

---
## Section 7: Reason Codes for Rejected Applicants

Each tree path from root → leaf records the conditions that led to the rejection decision.  
These are translated into plain English and used to generate RBI-compliant rejection notices.

In [ ]:
READABLE = {
    'checking_enc'    : 'Checking Account Status',
    'savings_enc'     : 'Savings Account Balance',
    'employment_enc'  : 'Employment Length',
    'ch_enc'          : 'Credit History',
    'job_enc'         : 'Job Category',
    'duration_months' : 'Loan Duration (months)',
    'credit_amount'   : 'Credit Amount (DM)',
    'installment_rate': 'Installment Rate (% of income)',
    'age'             : 'Applicant Age',
    'residence_years' : 'Years at Current Address',
    'num_existing_credits': 'Number of Existing Credits',
    'num_dependents'  : 'Number of Dependants',
    'telephone'       : 'Has Telephone Registration',
    'foreign_worker'  : 'Foreign Worker Status',
}

def get_reason_codes(tree_model, X_sample, feature_names, threshold=0.5):
    tree = tree_model.tree_
    results = []
    for i in range(len(X_sample)):
        node, path = 0, []
        sample = X_sample.iloc[i].values
        while tree.feature[node] != -2:
            fi   = tree.feature[node]
            name = feature_names[fi]
            tv   = tree.threshold[node]
            if sample[fi] <= tv:
                path.append(f"{name} <= {tv:.3f}")
                node = tree.children_left[node]
            else:
                path.append(f"{name} > {tv:.3f}")
                node = tree.children_right[node]
        vals     = tree.value[node][0]
        bad_prob = vals[1] / vals.sum()
        decision = "REJECTED" if bad_prob > threshold else "APPROVED"
        results.append({'idx': X_sample.index[i], 'prob': bad_prob,
                        'decision': decision, 'path': path})
    return results

# Sample 3 rejected applicants
rejected_idx = np.where(dt_pruned.predict(X_test)==1)[0][:3]
sample_rej   = X_test.iloc[rejected_idx]
rc_list      = get_reason_codes(dt_pruned, sample_rej, FEATURES, threshold=0.4)

print("SAMPLE REJECTION REASON CODES")
print("=" * 60)
for rc in rc_list:
    applicant = df.loc[rc['idx']]
    print(f"\nApplicant ID     : {rc['idx']}")
    print(f"Decision         : {rc['decision']}")
    print(f"P(Bad Credit)    : {rc['prob']:.1%}")
    print(f"Checking Account : {applicant['checking_account']}")
    print(f"Credit History   : {applicant['credit_history']}")
    print(f"Savings          : {applicant['savings_account']}")
    print(f"Decision Path:")
    for step, cond in enumerate(rc['path'], 1):
        human = cond
        for k, v in READABLE.items():
            human = human.replace(k, v)
        print(f"  Step {step}: {human}")
    print("-" * 60)

In [ ]:
rc = rc_list[0]
applicant = df.loc[rc['idx']]

letter = f"""
══════════════════════════════════════════════════════════
           SURAKSHA NBFC PVT. LTD.
           Credit Assessment Division
══════════════════════════════════════════════════════════
Application Ref  : NBFC-GC-{rc['idx']:05d}
Applicant Age    : {applicant['age']} years
Loan Purpose     : {applicant['purpose'].replace('_',' ').title()}
Amount Requested : DM {applicant['credit_amount']:,}
Duration         : {applicant['duration_months']} months

DECISION: APPLICATION DECLINED

Dear Applicant,

After a thorough assessment of your credit application, we
regret to inform you that your application has been declined.

REASONS FOR DECLINE:
"""

reason_map = {{
    'checking_enc': (
        'Checking Account Status',
        'Your checking account shows an insufficient or overdrawn balance. '
        'A minimum balance of 0 DM is required for loan eligibility.'),
    'savings_enc': (
        'Insufficient Savings',
        'Your savings account balance falls below our minimum threshold. '
        'Building savings of at least 100 DM is recommended before re-applying.'),
    'ch_enc': (
        'Adverse Credit History',
        'Your credit history includes delays or critical account status. '
        'Maintaining timely payments for 12+ months will improve your profile.'),
    'duration_months': (
        'Loan Duration',
        'The requested loan duration exceeds our standard lending parameters.'),
    'credit_amount': (
        'Credit Amount',
        'The requested credit amount is beyond your current repayment capacity.'),
    'employment_enc': (
        'Employment Stability',
        'Limited employment history increases repayment uncertainty. '
        'A minimum of 1 year of stable employment is preferred.'),
}}

for step, cond in enumerate(rc['path'], 1):
    for feat_key, (title, explanation) in reason_map.items():
        if feat_key in cond:
            letter += f"  {step}. {title}\n"
            letter += f"     {explanation}\n\n"
            break

letter += f"""
NEXT STEPS:
  • Improve checking and savings account balances
  • Ensure timely repayment of all existing credit obligations
  • Re-apply after 90 days once financial profile improves

REGULATORY DISCLOSURE:
  This decision was made using an automated credit scoring model
  per RBI Master Direction DNBR.PD.007/03.10.119/2016-17.
  You may request a manual review within 30 days.

  Grievance Officer: grievance@surakshafinance.in
══════════════════════════════════════════════════════════
"""
print(letter)

---
## Section 8 (Stretch): Fairness Audit — Disparate Impact Analysis

**Disparate Impact Ratio (DIR)** = Minority approval rate / Majority approval rate

**Four-Fifths Rule:** DIR < 0.80 → discriminatory under EEOC and most regulatory frameworks

In [ ]:
X_test_demo = X_test.copy()
X_test_demo['pred']     = dt_pruned.predict(X_test)
X_test_demo['actual']   = y_test.values
X_test_demo['approved'] = (X_test_demo['pred'] == 0).astype(int)
X_test_demo['gender']   = df_model.loc[X_test.index, 'is_female'].map({0:'Male',1:'Female'}).values
X_test_demo['age_bracket'] = df_model.loc[X_test.index, 'age_bracket'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('German Credit — Fairness Audit: Approval Rates', fontsize=13, fontweight='bold')

# Gender
gender_rates = X_test_demo.groupby('gender')['approved'].mean() * 100
dir_gender   = gender_rates.min() / gender_rates.max()
bars = axes[0].bar(gender_rates.index, gender_rates.values,
                   color=['#3498DB','#E67E22'], width=0.5, edgecolor='white')
axes[0].axhline(80*gender_rates.max()/100, color='red', linestyle='--', label='80% threshold')
for bar in bars:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
axes[0].set_title(f'By Gender  (DIR={dir_gender:.3f}  {"PASS ✓" if dir_gender>=0.8 else "FAIL ⚠"})',
                  fontweight='bold')
axes[0].set_ylabel('Approval Rate (%)'); axes[0].legend(); axes[0].set_ylim(0,100)

# Age bracket
age_rates = X_test_demo.groupby('age_bracket')['approved'].mean() * 100
dir_age   = age_rates.min() / age_rates.max()
bars2 = axes[1].bar(age_rates.index, age_rates.values,
                    color=['#E74C3C','#F39C12','#3498DB','#2ECC71'], width=0.6, edgecolor='white')
axes[1].axhline(80*age_rates.max()/100, color='red', linestyle='--', label='80% threshold')
for bar in bars2:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
axes[1].set_title(f'By Age Bracket  (DIR={dir_age:.3f}  {"PASS ✓" if dir_age>=0.8 else "FAIL ⚠"})',
                  fontweight='bold')
axes[1].set_ylabel('Approval Rate (%)'); axes[1].legend(); axes[1].set_ylim(0,100)

plt.tight_layout()
plt.savefig('gc_fairness.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Gender DIR : {dir_gender:.3f}  {'PASS ✓' if dir_gender>=0.8 else 'FAIL ⚠ — violates four-fifths rule'}")
print(f"Age DIR    : {dir_age:.3f}  {'PASS ✓' if dir_age>=0.8 else 'FAIL ⚠ — violates four-fifths rule'}")

---
## Level 2A: Bootstrap Stability — Are Reason Codes Reliable?

**Question:** If we retrain on slightly different data, do the root splits change?  
**Why it matters:** Unstable splits → unreliable reason codes → unauditable model.  
**Test:** Bootstrap 50 trees, record root and depth-2 splits.

In [ ]:
root_splits, d2_splits = [], []
for i in range(50):
    idx  = np.random.choice(len(X_train), len(X_train), replace=True)
    dt_b = DecisionTreeClassifier(ccp_alpha=optimal_alpha, random_state=i,
                                  class_weight='balanced')
    dt_b.fit(X_train.iloc[idx], y_train.iloc[idx])
    t = dt_b.tree_
    if t.feature[0] != -2:
        root_splits.append(FEATURES[t.feature[0]])
    for child in [t.children_left[0], t.children_right[0]]:
        if child != -1 and t.feature[child] != -2:
            d2_splits.append(FEATURES[t.feature[child]])

root_ctr = pd.Series(root_splits).value_counts()
d2_ctr   = pd.Series(d2_splits).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('German Credit — Bootstrap Stability (n=50)', fontsize=13, fontweight='bold')

colors_r = ['#2ECC71' if i==0 else '#3498DB' if i==1 else '#E67E22'
            for i in range(len(root_ctr))]
axes[0].bar(root_ctr.index, root_ctr.values/50*100, color=colors_r, edgecolor='white')
axes[0].axhline(70, color='red', linestyle='--', label='70% stability bar')
axes[0].set_title('Root Split Frequency'); axes[0].set_ylabel('% of Bootstrap Trees')
axes[0].legend(); axes[0].tick_params(axis='x', rotation=30)

d2_top = d2_ctr.head(6)
axes[1].bar(d2_top.index, d2_top.values/100*100, color='#9B59B6', edgecolor='white')
axes[1].set_title('Depth-2 Split Frequency')
axes[1].set_ylabel('% Frequency'); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('gc_bootstrap.png', bbox_inches='tight', dpi=150)
plt.show()

top_feat = root_ctr.index[0]
top_pct  = root_ctr.iloc[0]/50*100
print(f"Root split '{top_feat}' in {top_pct:.0f}% of trees")
if top_pct >= 70:
    print("STABLE ✓ — reason codes are reproducible and auditable.")
else:
    print("UNSTABLE ⚠ — reason codes may vary across retrains. Use stable alternatives.")

---
## Level 2B: Profitability Scoring — Business-Optimal Threshold

**Standard threshold = 0.5** ignores business economics.  
We assign real rupee values to each outcome and find the threshold that maximises total portfolio profit.

| Outcome | Value |
|---|---|
| Correctly approved good loan | +₹4,000 profit |
| Approved loan that defaults | −₹25,000 loss |

In [ ]:
PROFIT_GOOD  =  4000
LOSS_DEFAULT = -25000

proba_arr  = dt_pruned.predict_proba(X_test)[:,1]
thresholds = np.linspace(0.01, 0.99, 200)
profits, f1s, accs = [], [], []

for t in thresholds:
    preds = (proba_arr >= t).astype(int)
    profit = sum(PROFIT_GOOD if a==0 else LOSS_DEFAULT
                 for a, appr in zip(y_test, preds==0) if appr)
    profits.append(profit)
    f1s.append(f1_score(y_test, preds, zero_division=0))
    accs.append(accuracy_score(y_test, preds))

opt_t    = thresholds[np.argmax(profits)]
opt_f1_t = thresholds[np.argmax(f1s)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('German Credit — Profit-Optimal Threshold', fontsize=13, fontweight='bold')

axes[0].plot(thresholds, [p/1000 for p in profits], lw=2.5, color='#27AE60')
axes[0].axvline(opt_t, color='#27AE60', linestyle='--', lw=2, label=f'Profit-optimal t={opt_t:.2f}')
axes[0].axvline(0.5, color='gray', linestyle=':', lw=1.5, label='Default t=0.5')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Portfolio Profit (₹ thousands)')
axes[0].set_title('Profitability vs Threshold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, accs, lw=2, color='#3498DB', label='Accuracy')
axes[1].plot(thresholds, f1s,  lw=2, color='#E74C3C', label='F1 Score')
axes[1].axvline(opt_t, color='#27AE60', linestyle='--', lw=2, label=f'Profit-optimal t={opt_t:.2f}')
axes[1].set_xlabel('Threshold'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_title('Accuracy & F1 vs Threshold')

plt.tight_layout()
plt.savefig('gc_threshold.png', bbox_inches='tight', dpi=150)
plt.show()

def portfolio_profit(t):
    preds = (proba_arr >= t).astype(int)
    return sum(PROFIT_GOOD if a==0 else LOSS_DEFAULT
               for a, appr in zip(y_test, preds==0) if appr)

print(f"{'Strategy':<30} {'Threshold':>10} {'Portfolio Profit (₹)':>22}")
print("-"*65)
for name, t in [('Profit-maximising',opt_t),('F1-maximising',opt_f1_t),('Default (0.5)',0.5)]:
    print(f"{name:<30} {t:>10.2f} {portfolio_profit(t):>22,}")

---
## Level 2C: Auditor Walkthrough — 3 Loan Applications

For each application: decision path + reason codes + regulatory reference.

In [ ]:
proba_arr_all = dt_pruned.predict_proba(X_test)[:,1]
pred_arr      = (proba_arr_all >= opt_t).astype(int)

# One approved, one rejected, one borderline
idx_approved   = np.where((pred_arr==0) & (y_test.values==0))[0][0]
idx_rejected   = np.where((pred_arr==1) & (y_test.values==1))[0][0]
borderline_idx = np.abs(proba_arr_all - opt_t)
idx_borderline = np.argsort(borderline_idx)[0]

for case_idx, label in [(idx_approved,'APPROVED'),(idx_rejected,'REJECTED'),(idx_borderline,'BORDERLINE')]:
    app_id  = X_test.index[case_idx]
    prob    = proba_arr_all[case_idx]
    actual  = y_test.values[case_idx]
    decision = "APPROVED" if pred_arr[case_idx]==0 else "REJECTED"
    outcome  = "GOOD CREDIT (Repaid)" if actual==0 else "BAD CREDIT (Defaulted)"
    app_info = df.loc[app_id]

    rc = get_reason_codes(dt_pruned, X_test.iloc[[case_idx]], FEATURES, threshold=opt_t)[0]

    print(f"{'='*65}")
    print(f"  CASE: {label}")
    print(f"{'='*65}")
    print(f"  Application ID     : NBFC-GC-{app_id:05d}")
    print(f"  Checking Account   : {app_info['checking_account']}")
    print(f"  Credit History     : {app_info['credit_history']}")
    print(f"  Savings            : {app_info['savings_account']}")
    print(f"  Employment         : {app_info['employment_length']}")
    print(f"  Credit Amount      : DM {app_info['credit_amount']:,}")
    print(f"  Duration           : {app_info['duration_months']} months")
    print(f"  Age                : {app_info['age']}")
    print(f"  P(Bad Credit)      : {prob:.1%}")
    print(f"  Model Decision     : {decision}  (threshold={opt_t:.2f})")
    print(f"  Actual Outcome     : {outcome}")
    print(f"  Decision Path:")
    for step, cond in enumerate(rc['path'], 1):
        human = cond
        for k, v in READABLE.items():
            human = human.replace(k, v)
        print(f"    Step {step}: {human}")
    print(f"  Regulatory Ref     : RBI Master Direction DNBR.PD.007/03.10.119/2016-17")
    print()

---
## Section 9: Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#F8F9FA')
gs  = fig.add_gridspec(2, 4, hspace=0.45, wspace=0.4)
ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[0,2]); ax4 = fig.add_subplot(gs[0,3])
ax5 = fig.add_subplot(gs[1,0:2]); ax6 = fig.add_subplot(gs[1,2:4])

fig.suptitle('German Credit Dataset — Case Study 7 Summary Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

for ax, (val, lbl, col) in zip([ax1,ax2,ax3,ax4], [
    (f'{auc_dt:.3f}', 'Tree ROC-AUC', '#3498DB'),
    (f'{ks_dt:.3f}',  'KS Statistic', '#9B59B6'),
    (f'{opt_t:.2f}',  'Profit Threshold','#27AE60'),
    (f'{dir_gender:.3f}','Gender DIR','#E67E22')]):
    ax.set_facecolor(col)
    ax.text(0.5,0.6,val,ha='center',va='center',fontsize=28,color='white',
            fontweight='bold',transform=ax.transAxes)
    ax.text(0.5,0.2,lbl,ha='center',va='center',fontsize=11,color='white',
            transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

ax5.plot(fpr_lr, tpr_lr, lw=2, color='#3498DB', label=f'LR (AUC={auc_lr:.3f})')
ax5.plot(fpr_dt, tpr_dt, lw=2, color='#E67E22', label=f'DT (AUC={auc_dt:.3f})')
ax5.plot([0,1],[0,1],'k--',alpha=0.3)
ax5.set_xlabel('FPR'); ax5.set_ylabel('TPR')
ax5.set_title('ROC Curve Comparison', fontweight='bold')
ax5.legend(); ax5.grid(True, alpha=0.3)

importances = pd.Series(dt_pruned.feature_importances_, index=FEATURES)
importances = importances[importances>0].sort_values(ascending=True)
colors_imp  = ['#E74C3C' if v>importances.quantile(0.75) else
               '#3498DB' if v>importances.quantile(0.4) else '#95A5A6'
               for v in importances.values]
importances.plot(kind='barh', ax=ax6, color=colors_imp, edgecolor='white')
ax6.set_title('Feature Importances (Pruned Tree)', fontweight='bold')
ax6.set_xlabel('Importance (Gini)'); ax6.grid(True, alpha=0.3, axis='x')

plt.savefig('gc_dashboard.png', bbox_inches='tight', dpi=150, facecolor='#F8F9FA')
plt.show()

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY — German Credit Dataset")
print("="*60)
print(f"  Dataset         : UCI Statlog German Credit Data")
print(f"  Applicants      : {len(df):,}  |  Features: {len(FEATURES)}")
print(f"  Default rate    : {df['default'].mean():.1%}")
print(f"  Optimal alpha   : {optimal_alpha:.6f}")
print(f"  Tree depth      : {dt_pruned.get_depth()}")
print(f"  Tree leaves     : {dt_pruned.get_n_leaves()}")
print(f"  DT AUC          : {auc_dt:.4f}")
print(f"  LR AUC          : {auc_lr:.4f}")
print(f"  KS Statistic    : {ks_dt:.4f}")
print(f"  Profit threshold: {opt_t:.2f}")
print(f"  Gender DIR      : {dir_gender:.3f}  ({'PASS' if dir_gender>=0.8 else 'FAIL'})")
print(f"  Age DIR         : {dir_age:.3f}  ({'PASS' if dir_age>=0.8 else 'FAIL — younger applicants face higher rejection'})")